# XLM-R on RU22Fact: raw and artefact-removed evidence

Companion to the paper in [MHKhan50/CivIntel-AgentBench](https://github.com/MHKhan50/CivIntel-AgentBench). Fine-tunes `xlm-roberta-base` on claim and evidence pairs, three seeds, on the raw evidence and on the evidence with formatting artefacts removed.

Kaggle settings: **GPU T4**, **Internet on**. Run all cells (about one hour).

In [1]:
# Settings
REPO_URL = "https://github.com/MHKhan50/CivIntel-AgentBench.git"
REPO_REF = "main"
SEEDS = "0,1,2"
VARIANTS = "raw,cleaned"
OUT = "/kaggle/working/ru22fact-neural"

In [2]:
import subprocess, sys, torch
print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import transformers; print("transformers", transformers.__version__)

torch 2.11.0+cu130 CUDA: True
GPU: Tesla T4
transformers 5.20.2


In [3]:
# Code and data (RU22Fact is downloaded from its pinned commit and checked by SHA-256)
import os
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "repo"], check=True)
subprocess.run(["git", "clone", "-q", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-download", "--data-dir", "data/raw/ru22fact"], check=True)

07187d2bc477289d0a59baf5b515995657c18d65

{
  "data_dir": "data/raw/ru22fact",
  "sha256": {
    "train": "9dc0f12d5d3b1a4a88a437c5630f216a31196ffd32b3ffac5d815c41d4726db8",
    "validate": "9b6e6b38b478204a28229791fde50385854b12309f2d60a49159c7f9f159f658",
    "test": "946b548f511deefac6898aad02189c809653e017aeda0f1f439def830a8a3aaf"
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-download', '--data-dir', 'data/raw/ru22fact'], returncode=0)

In [4]:
# Smoke run (about a minute): catches errors before the long run
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--data-dir", "data/raw/ru22fact",
                "--output", "/kaggle/working/smoke", "--seeds", "0", "--variants", "raw", "--epochs", "1", "--limit", "20"], check=True)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2583.14it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 0, "best_epoch": 1, "minutes": 0.1, "test_macro_f1": 0.3301}
{
  "raw": {
    "seeds": [
      0
    ],
    "macro_f1": {
      "mean": 0.3301282051282051,
      "std": 0.0
    },
    "per_language": {
      "en": {
        "mean": 0.2853046594982079,
        "std": 0.0
      },
      "ru": {
        "mean": 0.0,
        "std": 0.0
      },
      "uk": {
        "mean": 0.4,
        "std": 0.0
      },
      "zh": {
        "mean": 0.20512820512820515,
        "std": 0.0
      }
    },
    "minutes": 0.1
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-neural', '--data-dir', 'data/raw/ru22fact', '--output', '/kaggle/working/smoke', '--seeds', '0', '--variants', 'raw', '--epochs', '1', '--limit', '20'], returncode=0)

In [5]:
# Full run: 2 variants x 3 seeds
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--data-dir", "data/raw/ru22fact",
                "--output", OUT, "--seeds", SEEDS, "--variants", VARIANTS], check=True)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2530.65it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 0, "best_epoch": 3, "minutes": 9.0, "test_macro_f1": 0.6918}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2385.20it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 1, "best_epoch": 2, "minutes": 8.9, "test_macro_f1": 0.6883}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2501.26it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "raw", "seed": 2, "best_epoch": 3, "minutes": 9.1, "test_macro_f1": 0.6856}


... (3 outputs left out)

{"variant": "cleaned", "seed": 1, "best_epoch": 2, "minutes": 8.8, "test_macro_f1": 0.6888}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2415.25it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "cleaned", "seed": 2, "best_epoch": 3, "minutes": 9.1, "test_macro_f1": 0.6964}
{
  "cleaned": {
    "seeds": [
      0,
      1,
      2
    ],
    "macro_f1": {
      "mean": 0.6910397463187383,
      "std": 0.004700206219760486
    },
    "per_language": {
      "en": {
        "mean": 0.6439518701745747,
        "std": 0.008019487784706325
      },
      "ru": {
        "mean": 0.9969388691106408,
        "std": 0.004771662433773905
      },
      "uk": {
        "mean": 0.9992301770592764,
        "std": 0.0006666862230826673
      },
      "zh": {
        "mean": 0.7004134619718575,
        "std": 0.024850822835251266
      }
    },
    "minutes": 9.0
  },
  "raw": {
    "seeds": [
      0,
      1,
      2
    ],
    "macro_f1": {
      "mean": 0.6885650509991389,
      "std": 0.0030886341731488215
    },
    "per_language": {
      "en": {
        "mean": 0.6427198206266623,
        "std": 0.005190529306480927
      },
      "ru": {
        "mean": 0.997375328083989

CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-neural', '--data-dir', 'data/raw/ru22fact', '--output', '/kaggle/working/ru22fact-neural', '--seeds', '0,1,2', '--variants', 'raw,cleaned'], returncode=0)

In [6]:
import json
summary = json.load(open(OUT + "/summary.json"))
for variant, s in summary.items():
    langs = ", ".join("{} {:.3f}".format(k, v["mean"]) for k, v in s["per_language"].items())
    print("{:8s} macro-F1 {:.3f} +/- {:.3f} | {} | {:.0f} min per seed".format(
        variant, s["macro_f1"]["mean"], s["macro_f1"]["std"], langs, s["minutes"]))

cleaned  macro-F1 0.691 +/- 0.005 | en 0.644, ru 0.997, uk 0.999, zh 0.700 | 9 min per seed
raw      macro-F1 0.689 +/- 0.003 | en 0.643, ru 0.997, uk 1.000, zh 0.676 | 9 min per seed
